# ML Model Training Evaluation on Processed UFC Master Data
Will now be performing a train-test split evaluation on multiple machine learning models on the processed data created from the previous notebook. The following ML models be evaluated based on their performance, from which the model with the highest accuracy will be selected:
- Logistic Regression
- Random Forest
- XGBoost
- Support Vector Machine

**Now loading our processed data:**

In [ ]:
from pathlib import Path
import pandas as pd

# Always calculate relative to notebook location
NOTEBOOK_DIR = Path.cwd()  # This is /.../ml_core/notebooks/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent  # Go up 2 levels: /.../ufc-fight-predictor/

# Build the data path to where processed training data source is located (ufc-master-processed.csv)
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "ufc-master-processed.csv"
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📄 Data path: {DATA_PATH}")

# Load data
df = pd.read_csv(DATA_PATH)


# Separating our X and y features
X = df.drop(columns=['Winner_Encoded']) # All our X features
y = df['Winner_Encoded'] # Our target


# Verifying size of original data, training set and test set
print("Total number of rows:")
print(f"Original data: {len(df)}")


📁 Project root: c:\Users\subsi\OneDrive\Desktop\ufc-fight-predictor
📄 Data path: c:\Users\subsi\OneDrive\Desktop\ufc-fight-predictor\data\processed\ufc-master-processed.csv
Total number of rows:
Original data: 6528


**Now evaluating the performance and accuracy of our first model, Logistic Regression**

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# First, scale the dataset to be used for the training models that need it (Logistic Regression and Support Vector Machine)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initializing and training the model
model = LogisticRegression(random_state=42, max_iter=5000)

# Cross-validation for accuracy comparison
score = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

# Fitting separately on full scaled data to extract feature importance
model.fit(X_scaled, y)

# Checking for which features played the biggest role
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.64701378 0.62557427 0.65620214 0.62605364 0.63908046]
Mean accuracy: 63.88%
Std dev: 1.19%


,feature,coefficient
2,RedExpectedValue,0.381257
26,BlueWeightLbs,0.241141
94,FinishRound,0.166265
40,RedWinsByDecisionSplit,0.149848
61,ReachDif,0.145454
...,...,...
15,BlueTotalRoundsFought,-0.162712
37,RedTotalRoundsFought,-0.166123
95,TotalFightTimeSecs,-0.229906
48,RedWeightLbs,-0.304222


**Now evaluating the accuracy and performance of our second model, Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Will not be using the scaled version of testing set as it's unnecessary
model = RandomForestClassifier(random_state=42, n_estimators=100)

score = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

model.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.67075038 0.66003063 0.66462481 0.64061303 0.67049808]
Mean accuracy: 66.13%
Std dev: 1.11%


,feature,importance
2,RedExpectedValue,0.043826
3,BlueExpectedValue,0.040034
1,BlueOdds,0.036917
0,RedOdds,0.034449
97,BlueDecOdds,0.023414
...,...,...
139,FinishDetails_Kicks,0.000000
137,FinishDetails_Keylock,0.000000
149,FinishDetails_Peruvian Necktie,0.000000
153,FinishDetails_Scarf Hold,0.000000


**Now checking for our third model, XGBoost**

In [ ]:
from xgboost import XGBClassifier


model = XGBClassifier(random_state=42, eval_metric="logloss")

score = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

model.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.67840735 0.65849923 0.64701378 0.62988506 0.64291188]
Mean accuracy: 65.13%
Std dev: 1.63%


,feature,importance
2,RedExpectedValue,0.201549
1,BlueOdds,0.036675
118,Finish_M-DEC,0.027192
116,Finish_DQ,0.018003
72,RHeavyweightRank,0.015861
...,...,...
157,FinishDetails_Spinning Back Kick,0.000000
159,FinishDetails_Triangle Armbar,0.000000
158,FinishDetails_Straight Armbar,0.000000
161,FinishDetails_Twister,0.000000


**Now testing our last model, Support Vector Machine**

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

# Will make use of scaled features, as it's necessary here since its distance-based
model = SVC(random_state=42, probability=False, kernel='linear')

score = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
print(f"Cross-val accuracy scores: {score}")
print(f"Mean accuracy: {score.mean():.2%}")
print(f"Std dev: {score.std():.2%}")

model.fit(X_scaled, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

display(feature_importance)

Cross-val accuracy scores: [0.66385911 0.643951   0.64777948 0.63295019 0.66283525]
Mean accuracy: 65.03%
Std dev: 1.17%


,feature,coefficient
3,BlueExpectedValue,1.092819
0,RedOdds,0.810499
11,BlueAvgTDLanded,0.165866
41,RedWinsByDecisionUnanimous,0.088535
64,AvgSubAttDif,0.086845
...,...,...
33,RedAvgTDLanded,-0.139039
45,RedWins,-0.148892
65,AvgTDDif,-0.174929
2,RedExpectedValue,-0.592517


Based on the observations, the model we will move ahead with is the Random Forest Classifier.

| Model | Mean Accuracy | Std Dev |
|---|---|---|
| Logistic Regression | 63.88% | 1.19% |
| Random Forest | 66.13% | 1.11% |
| XGBoost | 65.13% | 1.63% |
| SVM (linear) | 65.03% | 1.17% |

Random Forest had the highest average accuracy across the 5 folds while also having the lowest standard deviation, indicating both strong and consistent performance. This model will be retrained on the full dataset and exported for use in the backend API.